### Changes to the fairness analysis

- Added `stratify=y` to the train/test split so the training and test sets retain similar proportions of good- and bad-credit cases.
- Set the positive outcome label to `1` (good credit), so TPR and PPV are calculated for that outcome.
- Added predictive parity, measured as positive predictive value (PPV), alongside TPR.
- To evaluate more groups, add the feature name and its value-to-group mapping to `group_definitions`.

In [24]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import os
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144)

# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 

# metadata 
print(statlog_german_credit_data.metadata) 
  
# variable information 
print(statlog_german_credit_data.variables)

{'uci_id': 144, 'name': 'Statlog (German Credit Data)', 'repository_url': 'https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data', 'data_url': 'https://archive.ics.uci.edu/static/public/144/data.csv', 'abstract': 'This dataset classifies people described by a set of attributes as good or bad credit risks. Comes in two formats (one all numeric). Also comes with a cost matrix', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1000, 'num_features': 20, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Other', 'Marital Status', 'Age', 'Occupation'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1994, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5NC77', 'creators': ['Hans Hofmann'], 'intro_paper': None, 'additional_info': {'summary': 'Two datasets are provided.  the original dataset, in the form provided by

In [ ]:
column_names = {
    "Attribute1": "checking_status",
    "Attribute2": "duration_months",
    "Attribute3": "credit_history",
    "Attribute4": "purpose",
    "Attribute5": "credit_amount",
    "Attribute6": "savings",
    "Attribute7": "employment_since",
    "Attribute8": "installment_rate",
    "Attribute9": "personal_status_sex",
    "Attribute10": "other_debtors",
    "Attribute11": "residence_since",
    "Attribute12": "property",
    "Attribute13": "age_years",
    "Attribute14": "other_installment_plans",
    "Attribute15": "housing",
    "Attribute16": "existing_credits",
    "Attribute17": "job",
    "Attribute18": "dependents",
    "Attribute19": "telephone",
    "Attribute20": "foreign_worker",
}

group_definitions = {
    "Sex": (
        "personal_status_sex",
        {
            "A91": "Male",
            "A93": "Male",
            "A94": "Male",
            "A92": "Female",
            "A95": "Female",
        },
    ),
    "Foreign-worker status": (
        "foreign_worker",
        {
            "A201": "Foreign worker",
            "A202": "Not foreign worker",
        },
    ),
}

POSITIVE_LABEL = 1

X = X.rename(columns=column_names)
y = y.rename(columns={"class": "credit_risk"})

print(X.columns.tolist())
print(y.columns.tolist())

# Dataset-level label counts
y_series = y["credit_risk"].squeeze()
total_n = len(y_series)
total_pos = int((y_series == POSITIVE_LABEL).sum())
total_neg = int((y_series != POSITIVE_LABEL).sum())

print(f"\nTotal datapoints: {total_n}")
print(f"Total positive labels (credit_risk == {POSITIVE_LABEL}): {total_pos}")
print(f"Total negative labels (credit_risk != {POSITIVE_LABEL}): {total_neg}")

for attribute, (feature, mapping) in group_definitions.items():
    groups = X[feature].map(mapping)

    if groups.isna().any():
        unknown_values = X.loc[groups.isna(), feature].unique().tolist()
        raise ValueError(f"Unmapped values in {feature}: {unknown_values}")

    print(f"\n{attribute}:")
    for group_name in dict.fromkeys(mapping.values()):
        mask = groups == group_name
        n_group = int(mask.sum())
        n_pos_group = int((y_series[mask] == POSITIVE_LABEL).sum())
        n_neg_group = int((y_series[mask] != POSITIVE_LABEL).sum())
        print(
            f"  {group_name}: total={n_group}, "
            f"positive={n_pos_group}, negative={n_neg_group}"
        )

['checking_status', 'duration_months', 'credit_history', 'purpose', 'credit_amount', 'savings', 'employment_since', 'installment_rate', 'personal_status_sex', 'other_debtors', 'residence_since', 'property', 'age_years', 'other_installment_plans', 'housing', 'existing_credits', 'job', 'dependents', 'telephone', 'foreign_worker']
['credit_risk']

Total datapoints: 1000
Total positive labels (credit_risk == 1): 700
Total negative labels (credit_risk != 1): 300

Sensitive-group label distribution:

Sex:
  Male: total=690, positive=499, negative=191
  Female: total=310, positive=201, negative=109

Foreign-worker status:
  Foreign worker: total=963, positive=667, negative=296
  Not foreign worker: total=37, positive=33, negative=4


In [ ]:
# Define fairness measures as callable functions (no printing)

def calculate_tpr(y_true, y_pred, positive_label):
    actual_positive = y_true == positive_label
    if not actual_positive.any():
        return np.nan
    return np.mean(y_pred[actual_positive] == positive_label)

def calculate_ppv(y_true, y_pred, positive_label):
    predicted_positive = y_pred == positive_label
    if not predicted_positive.any():
        return np.nan
    return np.mean(y_true[predicted_positive] == positive_label)

def calculate_fpr(y_true, y_pred, positive_label):
    actual_negative = y_true != positive_label
    if not actual_negative.any():
        return np.nan
    return np.mean(y_pred[actual_negative] == positive_label)

def calculate_fnr(y_true, y_pred, positive_label):
    actual_positive = y_true == positive_label
    if not actual_positive.any():
        return np.nan
    return np.mean(y_pred[actual_positive] != positive_label)

def evaluate_fairness(y_test, y_pred, X_groups, positive_label, model_name="Model"):
    y_true = np.asarray(y_test).ravel()
    y_pred_flat = np.asarray(y_pred).ravel()

    if len(y_true) != len(X_groups) or len(y_pred_flat) != len(X_groups):
        raise ValueError("Length mismatch between labels/predictions and X_groups")

    results = []
    gaps = {}

    for attribute, (feature, mapping) in group_definitions.items():
        if feature not in X_groups.columns:
            raise KeyError(f"Required grouping feature '{feature}' not found in X_groups")

        groups = X_groups[feature].map(mapping)
        if groups.isna().any():
            unknown_values = X_groups.loc[groups.isna(), feature].unique().tolist()
            raise ValueError(f"Unmapped values in {feature}: {unknown_values}")

        group_names = list(dict.fromkeys(mapping.values()))
        group_tprs, group_ppvs, group_fprs, group_fnrs = {}, {}, {}, {}

        for group in group_names:
            mask = groups.to_numpy() == group
            tpr = calculate_tpr(y_true[mask], y_pred_flat[mask], positive_label)
            ppv = calculate_ppv(y_true[mask], y_pred_flat[mask], positive_label)
            fpr = calculate_fpr(y_true[mask], y_pred_flat[mask], positive_label)
            fnr = calculate_fnr(y_true[mask], y_pred_flat[mask], positive_label)

            group_tprs[group] = tpr
            group_ppvs[group] = ppv
            group_fprs[group] = fpr
            group_fnrs[group] = fnr

            results.append({
                "model": model_name,
                "attribute": attribute,
                "group": group,
                "TPR": tpr,
                "PPV": ppv,
                "FPR": fpr,
                "FNR": fnr,
            })

        valid_tprs = [rate for rate in group_tprs.values() if np.isfinite(rate)]
        valid_ppvs = [rate for rate in group_ppvs.values() if np.isfinite(rate)]
        valid_fprs = [rate for rate in group_fprs.values() if np.isfinite(rate)]
        valid_fnrs = [rate for rate in group_fnrs.values() if np.isfinite(rate)]

        gaps[(model_name, attribute)] = {
            "TPR": max(valid_tprs) - min(valid_tprs) if len(valid_tprs) >= 2 else np.nan,
            "PPV": max(valid_ppvs) - min(valid_ppvs) if len(valid_ppvs) >= 2 else np.nan,
            "FPR": max(valid_fprs) - min(valid_fprs) if len(valid_fprs) >= 2 else np.nan,
            "FNR": max(valid_fnrs) - min(valid_fnrs) if len(valid_fnrs) >= 2 else np.nan,
        }

    return pd.DataFrame(results), gaps

def format_two_result_tables(results1, results2, label1="Full", label2="Restricted", decimals=4):
    key_cols = ["attribute", "group"]
    metric_cols = ["TPR", "PPV", "FPR", "FNR"]

    left = results1[key_cols + metric_cols].copy()
    right = results2[key_cols + metric_cols].copy()

    merged = left.merge(right, on=key_cols, suffixes=("_1", "_2"), how="inner")

    out = merged[key_cols].copy()
    for m in metric_cols:
        out[m] = merged.apply(
            lambda r: f"{r[f'{m}_1']:.{decimals}f} ({r[f'{m}_2']:.{decimals}f})"
            if pd.notna(r[f"{m}_1"]) and pd.notna(r[f"{m}_2"])
            else "nan (nan)",
            axis=1,
        )

    out = out.rename(columns={m: f"{m} [{label1} ({label2})]" for m in metric_cols})
    return out

In [27]:
# Train two models: one with all features, one without sensitive features
sensitive_features = ['personal_status_sex', 'foreign_worker']

# Features for restricted model (without sensitive attributes)
X_restricted_features = [col for col in X.columns if col not in sensitive_features]

# Split the data
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_restricted = X_train_full[X_restricted_features]
X_test_restricted = X_test_full[X_restricted_features]
y_train_restricted = y_train_full
y_test_restricted = y_test_full

print('Training Model 1: Full features (includes sensitive attributes)')

categorical_features_full = X_train_full.select_dtypes(include=['object']).columns.tolist()
numerical_features_full = X_train_full.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor_full = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features_full),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_full)
    ])

pipeline_full = Pipeline(steps=[
    ('preprocessor', preprocessor_full),
    ('classifier', RandomForestClassifier(random_state=42))
])

pipeline_full.fit(X_train_full, y_train_full.values.ravel())
y_pred_full = pipeline_full.predict(X_test_full)
accuracy_full = pipeline_full.score(X_test_full, y_test_full)
print(f'Full Model Accuracy: {accuracy_full:.2f}')

print('\\nTraining Model 2: Restricted features (excludes sensitive attributes)')

categorical_features_restricted = X_train_restricted.select_dtypes(include=['object']).columns.tolist()
numerical_features_restricted = X_train_restricted.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor_restricted = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features_restricted),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_restricted)
    ])

pipeline_restricted = Pipeline(steps=[
    ('preprocessor', preprocessor_restricted),
    ('classifier', RandomForestClassifier(random_state=42))
])

pipeline_restricted.fit(X_train_restricted, y_train_restricted.values.ravel())
y_pred_restricted = pipeline_restricted.predict(X_test_restricted)
accuracy_restricted = pipeline_restricted.score(X_test_restricted, y_test_restricted)
print(f'Restricted Model Accuracy: {accuracy_restricted:.2f}')

Training Model 1: Full features (includes sensitive attributes)
Full Model Accuracy: 0.74
\nTraining Model 2: Restricted features (excludes sensitive attributes)
Restricted Model Accuracy: 0.77


In [31]:
results_full, gaps_full = evaluate_fairness(
    y_test_full, y_pred_full, X_test_full, POSITIVE_LABEL, "Full Model"
)

results_restricted, gaps_restricted = evaluate_fairness(
    y_test_restricted, y_pred_restricted, X_test_full, POSITIVE_LABEL, "Restricted Model"
)

comparison_df = pd.concat([results_full, results_restricted], ignore_index=True)
comparison_df


Full Model - Sex:
  TPR gap: 0.0209
  PPV gap: 0.0080
  FPR gap: 0.1000
  FNR gap: 0.0209

Full Model - Foreign-worker status:
  TPR gap: 0.1111
  PPV gap: 0.2357
  FPR gap: 0.6271
  FNR gap: 0.1111

Restricted Model - Sex:
  TPR gap: 0.0219
  PPV gap: 0.0216
  FPR gap: 0.0250
  FNR gap: 0.0219

Restricted Model - Foreign-worker status:
  TPR gap: 0.0667
  PPV gap: 0.0556
  FPR gap: 0.3898
  FNR gap: 0.0667


,model,attribute,group,TPR,PPV,FPR,FNR
0,Full Model,Sex,Male,0.898990,0.773913,0.650000,0.101010
1,Full Model,Sex,Female,0.878049,0.765957,0.550000,0.121951
2,Full Model,Foreign-worker status,Foreign worker,0.888889,0.764331,0.627119,0.111111
3,Full Model,Foreign-worker status,Not foreign worker,1.000000,1.000000,0.000000,0.000000
4,Restricted Model,Sex,Male,0.929293,0.786325,0.625000,0.070707
5,Restricted Model,Sex,Female,0.951220,0.764706,0.600000,0.048780
6,Restricted Model,Foreign-worker status,Foreign worker,0.933333,0.777778,0.610169,0.066667
7,Restricted Model,Foreign-worker status,Not foreign worker,1.000000,0.833333,1.000000,0.000000
